In [50]:
import mysql.connector
import time

# --- Configuration ---
db_config = {
    'host': '172.20.10.3',
    'user': 'ty69',         # Replace with your MySQL username
    'password': 'femboy', # Replace with your MySQL password
    'database': 'employees',
    'port' : 3306
}

In [51]:
# --- Helper Functions ---
def measure_query(cursor, query, test_name):
    """Executes a query and measures the time it takes to fetch all results."""
    print(f"Running: {test_name}...")
    
    # Start high-resolution timer
    start_time = time.time()
    
    cursor.execute(query)
    # We must fetch the rows to ensure the database actually processes the full result set
    results = cursor.fetchall() 
    
    end_time = time.time()
    duration = end_time - start_time
    
    print(f" -> Found {len(results)} rows.")
    print(f" -> Execution Time: {duration:.4f} seconds\n")
    return duration


In [52]:
# --- Helper Functions ---
def manage_index(cursor, action, index_name, table, columns=""):
    """Helper to create or drop indexes cleanly."""
    try:
        if action == "CREATE":
            print(f"Creating index '{index_name}' on {table}({columns})... This might take a moment.")
            cursor.execute(f"CREATE INDEX {index_name} ON {table}({columns})")
        elif action == "DROP":
            cursor.execute(f"DROP INDEX {index_name} ON {table}")
    except mysql.connector.Error as err:
        # Ignore errors if we try to drop an index that doesn't exist
        if action != "DROP": 
            print(f"Index Error: {err}")



In [53]:
# --- Main Test Suite ---
def run_tests():
    try:
        # Connect to Database
        conn = mysql.connector.connect(**db_config)
        cursor = conn.cursor()
        print("Connected to MySQL successfully!\n" + "="*40 + "\n")
        
        # TEST 4: Aggregation
        # ---------------------------------------------------------
        print("="*40)
        print("TEST 4: Aggregation")
        query_4 = "SELECT SQL_NO_CACHE emp_no, max(salary) as max_sal from salaries group by emp_no order by max_sal desc limit 10"
        
        manage_index(cursor, "DROP", "idx_agg_emp", "salaries")

        measure_query(cursor, query_4, "Aggregation Without Index")

        manage_index(cursor, "CREATE", "idx_agg_emp", "salaries", "emp_no")
        measure_query(cursor, query_4, "Aggregation With Index")

        # --- Clean up indexes we made during the test ---
        print("="*40)
        print("Cleaning up created test indexes...")
        manage_index(cursor, "DROP", "idx_hire_date_range", "employees")
        manage_index(cursor, "DROP", "idx_salaries_emp", "salaries")
        manage_index(cursor, "DROP", "idx_last_name", "employees")
        manage_index(cursor, "DROP", "idx_agg_emp", "salaries")
        print("Tests Complete!")

    except mysql.connector.Error as err:
        print(f"Error: {err}")
    finally:
        if 'conn' in locals() and conn.is_connected():
            cursor.close()
            conn.close()



In [ ]:
# --- Enhanced Test Suite with 5 Index Scenarios ---
def run_index_tests():
    """
    5 different indexing test scenarios:
    1. Simple Equality Filter (WHERE column = value)
    2. Range Query (WHERE column BETWEEN)
    3. JOIN + WHERE (Filter before join)
    4. ORDER BY with LIMIT (Top-N query)
    5. Multi-Column Index (Composite index)
    """
    try:
        conn = mysql.connector.connect(**db_config)
        cursor = conn.cursor()
        print("Connected to MySQL successfully!\n" + "="*50 + "\n")

        # ---------------------------------------------------------
        # TEST 1: Equality Filter (Single Column)
        # ---------------------------------------------------------
        print("TEST 1: EQUALITY FILTER (first_name = 'Steve')")
        print("-" * 50)
        query_1 = "SELECT SQL_NO_CACHE * FROM employees WHERE first_name = 'Steve'"
        
        manage_index(cursor, "DROP", "idx_test1_firstname", "employees")
        measure_query(cursor, query_1, "WITHOUT Index (Full Table Scan)")
        
        manage_index(cursor, "CREATE", "idx_test1_firstname", "employees", "first_name")
        measure_query(cursor, query_1, "WITH Index (Index Seek)")
        print()

        # ---------------------------------------------------------
        # TEST 2: Range Query (BETWEEN)
        # ---------------------------------------------------------
        print("TEST 2: RANGE QUERY (hire_date BETWEEN)")
        print("-" * 50)
        query_2 = """
            SELECT SQL_NO_CACHE * FROM employees 
            WHERE hire_date BETWEEN '1990-01-01' AND '1990-12-31'
        """
        
        manage_index(cursor, "DROP", "idx_test2_hiredate", "employees")
        measure_query(cursor, query_2, "WITHOUT Index (Range Scan)")
        
        manage_index(cursor, "CREATE", "idx_test2_hiredate", "employees", "hire_date")
        measure_query(cursor, query_2, "WITH Index (Range Index Seek)")
        print()

        # ---------------------------------------------------------
        # TEST 3: JOIN + WHERE (Multi-table filter)
        # ---------------------------------------------------------
        print("TEST 3: JOIN + WHERE (Filter before join)")
        print("-" * 50)
        query_3 = """
            SELECT SQL_NO_CACHE e.emp_no, e.first_name, e.last_name, s.salary
            FROM employees e
            JOIN salaries s ON e.emp_no = s.emp_no
            WHERE e.hire_date > '2000-01-01'
            LIMIT 100
        """
        
        manage_index(cursor, "DROP", "idx_test3_emp_hiredate", "employees")
        measure_query(cursor, query_3, "WITHOUT Index (Scan all + nested loop join)")
        
        manage_index(cursor, "CREATE", "idx_test3_emp_hiredate", "employees", "hire_date")
        measure_query(cursor, query_3, "WITH Index (Filter fast + efficient join)")
        print()

        # ---------------------------------------------------------
        # TEST 4: ORDER BY + LIMIT (Top-N query)
        # ---------------------------------------------------------
        print("TEST 4: ORDER BY + LIMIT (Top 100 highest salaries)")
        print("-" * 50)
        query_4 = """
            SELECT SQL_NO_CACHE emp_no, salary 
            FROM salaries 
            ORDER BY salary DESC 
            LIMIT 100
        """
        
        manage_index(cursor, "DROP", "idx_test4_salary_desc", "salaries")
        measure_query(cursor, query_4, "WITHOUT Index (Filesort)")
        
        manage_index(cursor, "CREATE", "idx_test4_salary_desc", "salaries", "salary")
        measure_query(cursor, query_4, "WITH Index (Pre-sorted index)")
        print()

        # ---------------------------------------------------------
        # TEST 5: Composite Index (Multi-column)
        # ---------------------------------------------------------
        print("TEST 5: COMPOSITE INDEX (last_name AND first_name)")
        print("-" * 50)
        query_5 = """
            SELECT SQL_NO_CACHE * FROM employees 
            WHERE last_name = 'Smith' AND first_name = 'John'
        """
        
        manage_index(cursor, "DROP", "idx_test5_composite", "employees")
        measure_query(cursor, query_5, "WITHOUT Index (Full table scan)")
        
        manage_index(cursor, "CREATE", "idx_test5_composite", "employees", "last_name, first_name")
        measure_query(cursor, query_5, "WITH Composite Index (Both columns)")
        print()

        # --- Cleanup ---
        print("="*50)
        print("Cleaning up test indexes...")
        manage_index(cursor, "DROP", "idx_test1_firstname", "employees")
        manage_index(cursor, "DROP", "idx_test2_hiredate", "employees")
        manage_index(cursor, "DROP", "idx_test3_emp_hiredate", "employees")
        manage_index(cursor, "DROP", "idx_test4_salary_desc", "salaries")
        manage_index(cursor, "DROP", "idx_test5_composite", "employees")
        print("All tests complete!\n")

    except mysql.connector.Error as err:
        print(f"Error: {err}")
    finally:
        if 'conn' in locals() and conn.is_connected():
            cursor.close()
            conn.close()


In [55]:
# Run the 5 index test scenarios
if __name__ == "__main__":
    run_index_tests()


Connected to MySQL successfully!

TEST 1: EQUALITY FILTER (first_name = 'Steve')
--------------------------------------------------
Running: WITHOUT Index (Full Table Scan)...
 -> Found 245 rows.
 -> Execution Time: 0.4365 seconds

Creating index 'idx_test1_firstname' on employees(first_name)... This might take a moment.
Running: WITH Index (Index Seek)...
 -> Found 245 rows.
 -> Execution Time: 0.0275 seconds


TEST 5: Order by with limit
Running: Order by with limit Without Index...
 -> Found 100 rows.
 -> Execution Time: 2.0909 seconds

Creating index 'idx_salary_limit' on salaries(salary)... This might take a moment.
Running: Order by with limit With Index...
 -> Found 100 rows.
 -> Execution Time: 0.0200 seconds

Cleaning up test indexes...
All tests complete!



In [56]:
if __name__ == "__main__":
    run_tests()

Connected to MySQL successfully!

TEST 4: Aggregation
Running: Aggregation Without Index...
 -> Found 10 rows.
 -> Execution Time: 2.8672 seconds

Creating index 'idx_agg_emp' on salaries(emp_no)... This might take a moment.
Running: Aggregation With Index...
 -> Found 10 rows.
 -> Execution Time: 2.2298 seconds

Cleaning up created test indexes...
Tests Complete!
